In [ ]:
#import zipfile
import os

file_path_A_first = os.getcwd() + "\\Exp-A_HDF5_Run-1.zip"
file_path_B_first = os.getcwd() + "\\Exp-B_HDF5_Run-1.zip"
file_path_C_first = os.getcwd() + "\\Exp-C_HDF5_Run-1.zip"

file_path_A_last = os.getcwd() + "\\Exp-A_HDF5_Run-5.zip"
file_path_B_last = os.getcwd() + "\\Exp-B_HDF5_Run-7.zip"
file_path_C_last = os.getcwd() + "\\Exp-C_HDF5_Run-12.zip"

#with zipfile.ZipFile(file_path, "r") as zip_ref:
    #print(zip_ref.namelist())

# sample size of dataset

In [31]:
import zipfile
import h5py
import numpy as np
import io

def extract_features_from_zip(zip_path, hdf5_filename, downsample=10):
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(hdf5_filename) as f:
            file_bytes = f.read()
            file_obj = io.BytesIO(file_bytes)

            with h5py.File(file_obj, 'r') as hdf:

                rms = hdf["CI/rms"][:]
                kurtosis = hdf["CI/kurtosis"][:]
                speed = hdf["Context/PAU Speed"][:]
                torque = hdf["Context/PAU Torque"][:]

                # Downsampling
                rms = rms[::downsample]
                kurtosis = kurtosis[::downsample]
                speed = speed[::downsample]
                torque = torque[::downsample]

                features = np.stack([rms, kurtosis, speed, torque], axis=1)
    return features

def find_file_in_zip(zip_path, target_name):
    with zipfile.ZipFile(zip_path, 'r') as z:
        for name in z.namelist():
            if target_name in name:
                return name
    return None






In [ ]:
import re
#keep gears isolated
 
def run_number_from_zip(zip_path):
    m = re.search(r"Run-(\d+)\.zip$", zip_path)
    return int(m.group(1)) if m else None


paths = [
    file_path_A_first, file_path_B_first, file_path_C_first,
    file_path_A_last,  file_path_B_last,  file_path_C_last
]

path_to_gear = {
    file_path_A_first: "gear1",
    file_path_B_first: "gear2",
    file_path_C_first: "gear3",
    
    file_path_A_last: "gear1",
    file_path_B_last: "gear2",
    file_path_C_last: "gear3",
}


path_to_prefix = {
    file_path_A_first: "Dyno Gear233Run1_",
    file_path_B_first: "Dyno Gear303Run1_",
    file_path_C_first: "Dyno Gear303Run1_",

    file_path_A_last:  "Dyno Gear233Run5_",
    file_path_B_last:  "Dyno Gear303Run7_",
    file_path_C_last:  "Dyno Gear303Run12_",
}



data = {
    "gear1": {"first": None, "last": None},
    "gear2": {"first": None, "last": None},
    "gear3": {"first": None, "last": None},
}



for path in paths:
    gear_key = path_to_gear[path]
    run_label = run_number_from_zip(path)
    prefix = path_to_prefix[path]

    all_features = []
    for i in range(0, 21):
        filename = f"{prefix}{i:05d}.hdf5"
        real_path = find_file_in_zip(path, filename)

        # optional robust: falls Datei fehlt
        if real_path is None:
            print(f"[WARN] Not found in {path}: {filename}")
            continue

        features = extract_features_from_zip(path, real_path)
        all_features.append(features)

    if not all_features:
        print(f"[WARN] No features extracted for {gear_key} {run_label}")
        continue

    sample_dataset = np.concatenate(all_features, axis=0)
    print(gear_key, run_label, sample_dataset.shape)

    data[gear_key][run_label] = sample_dataset



gear1 (123, 4)
gear2 (123, 4)


ValueError: all input arrays must have the same shape

## sliding windows

In [ ]:
import numpy as np

def make_windows(seq, window_size=20, stride=1):
    X = []
    for i in range(0, len(seq) - window_size + 1, stride):
        X.append(seq[i:i+window_size])
    return np.asarray(X)  # (n_windows, window_size, n_features)

def flatten_windows(X):
    return X.reshape(X.shape[0], -1)  # (n_samples, window_size*n_features)

def stats_features(X):
    # X: (n_samples, window_size, n_features)
    mean = X.mean(axis=1)
    std  = X.std(axis=1)
    mx   = X.max(axis=1)
    mn   = X.min(axis=1)
    return np.concatenate([mean, std, mx, mn], axis=1)  # (n_samples, 4*n_features)


def window_features_stats(Xw):
    # Xw: (n_samples, window, n_features)
    mean = Xw.mean(axis=1)
    std  = Xw.std(axis=1)
    mn   = Xw.min(axis=1)
    mx   = Xw.max(axis=1)

    # Trend (lineare Steigung) je Feature
    t = np.arange(Xw.shape[1])
    t = (t - t.mean()) / (t.std() + 1e-9)
    slope = (Xw * t[None, :, None]).mean(axis=1)  # proportional zur Steigung

    return np.concatenate([mean, std, mn, mx, slope], axis=1)


In [ ]:
#dataset from multiple gears/runs, keep order
import numpy as np

def build_window_dataset(data_dict, gears, window_size, stride, label_mode, feature_mode):
    X_all, y_all = [], []
    meta = []  # (gear_id, run_id, window_index)

    for gear_id in gears:
        runs = data_dict[gear_id]
        for run_id, seq in enumerate(runs):
            # 1) windows within THIS run only
            Xw = make_windows(seq, window_size=window_size, stride=stride)

            # 2) proxy labels within THIS run (or within gear)
            y = create_proxy_labels(len(Xw), mode=label_mode)

            # 3) features
            if feature_mode == "flat":
                X = flatten_windows(Xw)
            elif feature_mode == "stats":
                X = window_features_stats(Xw)
            else:
                raise ValueError("feature_mode must be 'flat' or 'stats'")

            X_all.append(X)
            y_all.append(y)
            meta += [(gear_id, run_id, i) for i in range(len(y))]

    X_all = np.vstack(X_all)
    y_all = np.concatenate(y_all)
    meta = np.array(meta, dtype=object)
    return X_all, y_all, meta

In [ ]:
def build_window_dataset(data_dict, gears, window_size=20, stride=1,
                         feature_mode="stats",
                         label_strategy="hi"):
    X_all, y_all, meta = [], [], []

    for gear_id in gears:
        runs = data_dict[gear_id]  # erwartet: [run_first, run_last] oder mehr
        for run_id, seq in enumerate(runs):

            run_kind = "first" if run_id == 0 else ("last" if run_id == len(runs)-1 else "mid")

            Xw = make_windows(seq, window_size=window_size, stride=stride)

            if feature_mode == "stats":
                X_feat = window_features_stats(Xw)
            else:
                # flat windows (eher selten sinnvoll hier)
                X_feat = Xw.reshape(Xw.shape[0], -1)

            # Labels
            if label_strategy == "binary":
                y = create_proxy_labels_binary(len(X_feat), run_kind=run_kind)
            elif label_strategy == "soft":
                y = create_proxy_labels_soft(len(X_feat), run_kind=run_kind)
            elif label_strategy == "hi":
                y = create_proxy_labels_hi(X_feat)
            else:
                raise ValueError("label_strategy must be 'binary', 'soft', or 'hi'")

            X_all.append(X_feat)
            y_all.append(y)
            meta += [(gear_id, run_id, run_kind, i) for i in range(len(y))]

    X_all = np.vstack(X_all)
    y_all = np.concatenate(y_all)
    meta  = np.array(meta, dtype=object)
    return X_all, y_all, meta

In [ ]:
#gear isloated split
#train on two gears, tes on one

train_gears = ["gear1", "gear2"]
test_gears  = ["gear3"]

X_train, y_train, meta_train = build_window_dataset(data, train_gears, window_size, stride, label_mode, feature_mode)
X_test,  y_test,  meta_test  = build_window_dataset(data, test_gears,  window_size, stride, label_mode, feature_mode)

#test gear is fully isolated: no shared samples, no shared windows, no shared scaling

## proxy labels
### Health indicator–driven proxy labels
In the absence of explicit failure labels, proxy labels are constructed based on the temporal ordering of samples within a run-to-failure experiment.
These labels are constructed based on the first and last run of the gears. This provides end-of-life (failure) signal, helps define a real damage scale and
covers nonlinear region

In [ ]:
def create_health_index(features):
    rms = features[:, 0]
    kurt = features[:, 1]

    # normalize per run
    rms_n = (rms - rms.min()) / (rms.max() - rms.min() + 1e-8)
    kurt_n = (kurt - kurt.min()) / (kurt.max() - kurt.min() + 1e-8)

    # combine
    health = 0.6 * rms_n + 0.4 * kurt_n
    return health

def extract_health_indicators(X_feat):
    # X_feat shape: (n_windows, 20) wenn 4 features * 5 stats
    rms_mean  = X_feat[:, 0]
    kurt_mean = X_feat[:, 1]
    rms_slope  = X_feat[:, 16]
    kurt_slope = X_feat[:, 17]
    return rms_mean, kurt_mean, rms_slope, kurt_slope

In [ ]:
import numpy as np

#Ranking-basierte Normalisierung (robust gegen Skalenunterschiede).

def minmax01(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def rank01(x):
    # ohne scipy: rank über argsort (ties ok-ish)
    order = np.argsort(x)
    ranks = np.empty_like(order, dtype=np.float32)
    ranks[order] = np.arange(len(x), dtype=np.float32)
    return ranks / (len(x) - 1 + 1e-8)

def create_health_index_from_window_features(X_feat, mode="rank", w=(0.5, 0.3, 0.1, 0.1), monotonic=True):
    rms_mean, kurt_mean, rms_slope, kurt_slope = extract_health_indicators(X_feat)

    # (A) innerhalb des Runs normalisieren
    if mode == "minmax":
        a = minmax01(rms_mean)
        b = minmax01(kurt_mean)
        c = minmax01(np.maximum(rms_slope, 0))   # nur positive Trends als „Damage“
        d = minmax01(np.maximum(kurt_slope, 0))
    elif mode == "rank":
        a = rank01(rms_mean)
        b = rank01(kurt_mean)
        c = rank01(np.maximum(rms_slope, 0))
        d = rank01(np.maximum(kurt_slope, 0))
    else:
        raise ValueError("mode must be 'rank' or 'minmax'")

    hi = w[0]*a + w[1]*b + w[2]*c + w[3]*d

    # (B) optional monotonic smoothing (HI soll nicht „gesünder“ werden)
    if monotonic:
        hi = np.maximum.accumulate(hi)

    # final 0..1
    hi = minmax01(hi)
    return hi


In [ ]:
def create_proxy_labels_hi(X_feat):
    return create_health_index_from_window_features(X_feat, mode="rank", monotonic=True)

In [ ]:
y = create_health_index(window_level_features)

#Ranking-based labels (robust to scaling differences)
#Instead of absolute values → use ordering within each run: 0 → healthiest, 1 → most damaged
# incase distributions differ across gears, signal scale varies
y = rankdata(health_indicator) / len(health_indicator)


In [ ]:
#X = create_windows(sample_dataset, window_size=20)
Xw = make_windows(seq)
X_feat = window_features_stats(Xw)

# create label from features of THIS run
y = create_health_index(X_feat)

# normalize within run
y = (y - y.min()) / (y.max() - y.min() + 1e-8)

## Model Training and Testing
Random forest regression

In [ ]:
def time_split(X, y, test_size=0.2): #with stride=1 and window_size=20, consecutive windows share 19/20 samples
    n = len(y)
    cut = int(n * (1 - test_size))
    return X[:cut], X[cut:], y[:cut], y[cut:]

#leakage caused by overlapping windows
#trying to prevent too similar test and training set
def time_split_with_gap(X, y, test_size=0.2, gap=20): 
    n = len(X)
    n_test = int(n * test_size)
    split = n - n_test

    # [split-gap, split+gap)
    train_end = max(0, split - gap)
    test_start = min(n, split + gap)

    X_train, y_train = X[:train_end], y[:train_end]
    X_test,  y_test  = X[test_start:], y[test_start:]

    return X_train, X_test, y_train, y_test

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR

def make_model(name="rf", **kwargs):
    if name == "rf":
        return RandomForestRegressor(
            n_estimators=300, random_state=42, n_jobs=-1, **kwargs
        )
    if name == "ridge":
        return Ridge(**kwargs)
    if name == "svr":
        return SVR(**kwargs)
    raise ValueError(f"Unknown model: {name}")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def run_experiment(sample_dataset,
                   window_size=20,
                   stride=1,
                   label_mode="linear",
                   feature_mode="flat",   # "flat" oder "stats"
                   model_name="rf",
                   test_size=0.2,
                   model_params=None):

    # 1) windows
    Xw = make_windows(sample_dataset, window_size=window_size, stride=stride)

    # 2) labels
    y = create_proxy_labels(len(Xw), mode=label_mode)

    # 3) features
    if feature_mode == "flat":
        X = flatten_windows(Xw)
    elif feature_mode == "stats":
        X = window_features_stats(Xw)
    else:
        raise ValueError("feature_mode must be 'flat' or 'stats'")

    # 4) split
    X_train, X_test, y_train, y_test = time_split_with_gap(X, y, test_size=test_size)

    # 5) model
    model_params = model_params or {}
    model = make_model(model_name, **model_params)
    model.fit(X_train, y_train)

    # 6) eval
    y_pred = model.predict(X_test)
    metrics = {
        "mse": mean_squared_error(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "r2":  r2_score(y_test, y_pred),
        "n_train": len(y_train),
        "n_test": len(y_test),
    }

    y_base = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
    print("Baseline MSE:", mean_squared_error(y_test, y_base))
    print("Model MSE:", mean_squared_error(y_test, y_pred))

    return model, metrics, (y_test, y_pred)

In [16]:
configs = [
    ("linear", "rf",   "flat"),
    ("quadratic", "rf","flat"),
    ("log", "rf",      "flat"),
    ("linear", "ridge","stats"),
    ("linear", "svr",  "stats"),
]

results = []
for label_mode, model_name, feat_mode in configs:
    _, m, _ = run_experiment(
        sample_dataset,
        window_size=20,
        label_mode=label_mode,
        feature_mode=feat_mode,
        model_name=model_name
    )
    results.append((label_mode, model_name, feat_mode, m))

results

Baseline MSE: 0.2584502755350506
Model MSE: 0.04000163230869703
Baseline MSE: 0.37817201326193656
Model MSE: 0.09965307023694857
Baseline MSE: 0.05506936981992142
Model MSE: 0.0030151300190924737
Baseline MSE: 0.2584502755350506
Model MSE: 0.06548313487840107
Baseline MSE: 0.2584502755350506
Model MSE: 0.16663457756882796


[('linear',
  'rf',
  'flat',
  {'mse': 0.04000163230869703,
   'mae': 0.18850451291627796,
   'r2': -10.35028134199138,
   'n_train': 82,
   'n_test': 21}),
 ('quadratic',
  'rf',
  'flat',
  {'mse': 0.09965307023694857,
   'mae': 0.2969685804605071,
   'r2': -7.681819278257791,
   'n_train': 82,
   'n_test': 21}),
 ('log',
  'rf',
  'flat',
  {'mse': 0.0030151300190924737,
   'mae': 0.052720498058848604,
   'r2': -14.18689892297396,
   'n_train': 82,
   'n_test': 21}),
 ('linear',
  'ridge',
  'stats',
  {'mse': 0.06548313487840107,
   'mae': 0.25071470964289233,
   'r2': -17.580541871133224,
   'n_train': 82,
   'n_test': 21}),
 ('linear',
  'svr',
  'stats',
  {'mse': 0.16663457756882796,
   'mae': 0.4038781872733927,
   'r2': -46.281803955256905,
   'n_train': 82,
   'n_test': 21})]

In [23]:
print(type(results))
print(len(results))
print(results)

<class 'list'>
5
[('linear', 'rf', 'flat', {'mse': 0.04000163230869703, 'mae': 0.18850451291627796, 'r2': -10.35028134199138, 'n_train': 82, 'n_test': 21}), ('quadratic', 'rf', 'flat', {'mse': 0.09965307023694857, 'mae': 0.2969685804605071, 'r2': -7.681819278257791, 'n_train': 82, 'n_test': 21}), ('log', 'rf', 'flat', {'mse': 0.0030151300190924737, 'mae': 0.052720498058848604, 'r2': -14.18689892297396, 'n_train': 82, 'n_test': 21}), ('linear', 'ridge', 'stats', {'mse': 0.06548313487840107, 'mae': 0.25071470964289233, 'r2': -17.580541871133224, 'n_train': 82, 'n_test': 21}), ('linear', 'svr', 'stats', {'mse': 0.16663457756882796, 'mae': 0.4038781872733927, 'r2': -46.281803955256905, 'n_train': 82, 'n_test': 21})]


## Visualization

In [27]:
import matplotlib.pyplot as plt
import numpy as np

def plot_true_vs_pred(y_test, y_pred, title="True vs Pred"):
    plt.figure(figsize=(10,4))
    plt.plot(y_test, marker="o", label="True")
    plt.plot(y_pred, marker="o", label="Pred")
    plt.title(title)
    plt.xlabel("Test index (time order)")
    plt.ylabel("Proxy label")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [34]:
'''results.append((transform, algo, feats, metrics, (y_test, y_pred)))
best = min(results, key=lambda r: r[3]["mse"])   # r[3] ist metrics
transform, algo, feats, metrics, (y_test, y_pred) = best
plot_true_vs_pred(y_test, y_pred)'''

'results.append((transform, algo, feats, metrics, (y_test, y_pred)))\nbest = min(results, key=lambda r: r[3]["mse"])   # r[3] ist metrics\ntransform, algo, feats, metrics, (y_test, y_pred) = best\nplot_true_vs_pred(y_test, y_pred)'